# CF Statement — First Tag Frequency

For each Cash Flow statement in `test_annual.db`, find the tag with the lowest `line` value (i.e. the first tag presented), then count how often each tag appears in that role.

In [16]:
import sqlite3
import pandas as pd

con = sqlite3.connect('../test_db/test_annual.db')

In [17]:
query = """
WITH cf_min_line AS (
    SELECT cik, ddate, MIN(line) AS min_line
    FROM facts
    WHERE stmt = 'CF'
    GROUP BY cik, ddate
),
first_tags AS (
    SELECT f.tag, f.label, f.cik, f.ddate, f.version, f.datatype, f.inpth, f.value
    FROM facts f
    JOIN cf_min_line m
      ON f.cik = m.cik
     AND f.ddate = m.ddate
     AND f.line = m.min_line
    WHERE f.stmt = 'CF'
),
counts AS (
    SELECT tag, label, COUNT(*) AS filing_count
    FROM first_tags
    GROUP BY tag, label
),
examples AS (
    SELECT tag, cik AS example_cik, ddate AS example_ddate, version AS example_version, datatype AS example_datatype, inpth AS example_inpth, value AS example_value,
           ROW_NUMBER() OVER (PARTITION BY tag ORDER BY cik) AS rn
    FROM first_tags
)
SELECT c.tag, c.label, c.filing_count,
       e.example_cik, e.example_ddate, e.example_version, e.example_datatype, e.example_inpth, e.example_value
FROM counts c
JOIN examples e ON c.tag = e.tag AND e.rn = 1
ORDER BY c.filing_count DESC
"""

df = pd.read_sql_query(query, con)
con.close()
df

,tag,label,filing_count,example_cik,example_ddate,example_version,example_datatype,example_inpth,example_value
0,NetIncomeLoss,Net Income (Loss) Attributable to Parent,408,1000230,20241031,us-gaap/2025,monetary,0,-4.210211e+06
1,ProfitLoss,"Net Income (Loss), Including Portion Attributa...",286,1001171,20241231,us-gaap/2024,monetary,0,1.954000e+06
2,IncomeLossFromContinuingOperationsIncludingPor...,"Income (Loss) from Continuing Operations, Net ...",15,1319161,20240930,us-gaap/2025,monetary,0,4.780000e+08
3,CashCashEquivalentsRestrictedCashAndRestricted...,"Cash, Cash Equivalent, Restricted Cash, and Re...",9,1376986,20240930,us-gaap/2025,monetary,0,5.230000e+08
4,CashAndCashEquivalentsAtCarryingValue,Cash and Cash Equivalent,7,1868878,20240930,us-gaap/2025,monetary,1,1.014700e+07
5,IncomeLossFromContinuingOperations,"Income (Loss) from Continuing Operations, Net ...",7,1703073,20240630,us-gaap/2024,monetary,0,9.809510e+05
6,CashHeldInForeignCurrencyAcquisitionCost,"Cash and Cash Equivalent, Held in Foreign Curr...",6,1476765,20240930,us-gaap/2025,monetary,1,7.973000e+06
7,ProfitAndLoss,[Net income (Loss)],6,1348362,20240831,0001640334-25-002225,monetary,0,5.808654e+06
8,NetIncomeLossAvailableToCommonStockholdersBasic,Net Income (Loss) Available to Common Stockhol...,4,1071840,20240831,us-gaap/2025,monetary,0,-3.055415e+06
9,ProceedsFromIncomeTaxRefunds,Proceeds from Income Tax Refunds,4,814547,20240930,us-gaap/2025,monetary,1,8.590000e+05


In [18]:

df[df['example_inpth'] == '0']

,tag,label,filing_count,example_cik,example_ddate,example_version,example_datatype,example_inpth,example_value
0,NetIncomeLoss,Net Income (Loss) Attributable to Parent,408,1000230,20241031,us-gaap/2025,monetary,0,-4210211.0
1,ProfitLoss,"Net Income (Loss), Including Portion Attributa...",286,1001171,20241231,us-gaap/2024,monetary,0,1954000.0
2,IncomeLossFromContinuingOperationsIncludingPor...,"Income (Loss) from Continuing Operations, Net ...",15,1319161,20240930,us-gaap/2025,monetary,0,478000000.0
3,CashCashEquivalentsRestrictedCashAndRestricted...,"Cash, Cash Equivalent, Restricted Cash, and Re...",9,1376986,20240930,us-gaap/2025,monetary,0,523000000.0
5,IncomeLossFromContinuingOperations,"Income (Loss) from Continuing Operations, Net ...",7,1703073,20240630,us-gaap/2024,monetary,0,980951.0
7,ProfitAndLoss,[Net income (Loss)],6,1348362,20240831,0001640334-25-002225,monetary,0,5808654.0
8,NetIncomeLossAvailableToCommonStockholdersBasic,Net Income (Loss) Available to Common Stockhol...,4,1071840,20240831,us-gaap/2025,monetary,0,-3055415.0
10,ProceedsFromSaleOfTrustAssetsToPayExpenses,Proceeds from Sale of Trust Assets to Pay Expe...,4,1222333,20240930,us-gaap/2025,monetary,0,236433000.0
12,AdjustmentForLongTermIntercompanyTransactionsG...,Adjustment for Long-Term Intra-Entity Transact...,2,2016167,20240930,us-gaap/2025,monetary,0,995270.0
16,NetIncomeLossNet,clfd_NetIncomeLossNet,2,796505,20240930,0001171843-25-007594,monetary,0,-12453000.0


In [19]:
df[df['example_inpth'] == '1']

,tag,label,filing_count,example_cik,example_ddate,example_version,example_datatype,example_inpth,example_value
4,CashAndCashEquivalentsAtCarryingValue,Cash and Cash Equivalent,7,1868878,20240930,us-gaap/2025,monetary,1,1.014700e+07
6,CashHeldInForeignCurrencyAcquisitionCost,"Cash and Cash Equivalent, Held in Foreign Curr...",6,1476765,20240930,us-gaap/2025,monetary,1,7.973000e+06
9,ProceedsFromIncomeTaxRefunds,Proceeds from Income Tax Refunds,4,814547,20240930,us-gaap/2025,monetary,1,8.590000e+05
11,DividendsPerShareDeclared,"Dividends, Per Share, Declared",3,1260221,20240930,0001260221-25-000081,perShare,1,7.500000e+01
13,AmortizationOfIntangibleAssets,Amortization of Intangible Assets,2,717954,20240831,us-gaap/2025,monetary,1,1.880000e+07
14,CashAndCashEquivalentsIncludingDiscontinuedOpe...,Cash And Cash Equivalents Including Discontinu...,2,1728688,20240930,0001728688-25-000122,monetary,1,8.654100e+07
15,InterestPaidCapitalized,"Interest Paid, Capitalized, Investing Activity",2,1547459,20240930,us-gaap/2025,monetary,1,3.380000e+05
17,NonCashEquivalentSegregatedAssets,Non-Cash Equivalent Segregated Assets,2,913760,20240930,0000913760-25-000196,monetary,1,5.180000e+07
19,AccountsPayableAndAccruedLiabilitiesCurrent,"Accounts Payable and Accrued Liabilities, Current",1,1845459,20241231,us-gaap/2024,monetary,1,1.566800e+07
20,BusinessCombinationRecognizedIdentifiableAsset...,"Business Combination, Recognized Asset Acquire...",1,1325670,20221231,us-gaap/2023,monetary,1,2.446000e+06
